In [3]:
import json
import requests

url = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json"

response = requests.get(url)
data = json.loads(response.text)

entry = data[0]

print(f"--- Data Exploration (Entry 0) ---")
print(f"Question: {entry['qa']['question']}")
print(f"Program: {entry['qa']['program']}")
print(f"Execution Answer: {entry['qa']['exe_ans']}")
print(f"Gold Evidence (Indices): {entry['qa']['gold_inds']}")
print(f"Table Row 0: {entry['table'][0]}")
print(f"Pre-text sample: {entry['pre_text'][:2]}")
print()

--- Data Exploration (Entry 0) ---
Question: what is the net change in net revenue during 2015 for entergy corporation?
Program: subtract(5829, 5735)
Execution Answer: 94.0
Gold Evidence (Indices): {'table_1': 'the 2014 net revenue of amount ( in millions ) is $ 5735 ;', 'table_8': 'the 2015 net revenue of amount ( in millions ) is $ 5829 ;'}
Table Row 0: ['', 'amount ( in millions )']
Pre-text sample: ['entergy corporation and subsidiaries management 2019s financial discussion and analysis a result of the entergy louisiana and entergy gulf states louisiana business combination , results of operations for 2015 also include two items that occurred in october 2015 : 1 ) a deferred tax asset and resulting net increase in tax basis of approximately $ 334 million and 2 ) a regulatory liability of $ 107 million ( $ 66 million net-of-tax ) as a result of customer credits to be realized by electric customers of entergy louisiana , consistent with the terms of the stipulated settlement in the b

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
print(f"Is CUDA available? {torch.cuda.is_available()}")

print(f"Model is on device: {model.device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Is CUDA available? True
Model is on device: cuda:0
GPU Name: Tesla T4


In [ ]:
import json
import requests
import random
import re
from tqdm import tqdm

def format_table(table):
    """Linearizes the table rows into a string format."""
    table_str = ""
    for row in table:
        table_str += " | ".join([str(cell) for cell in row]) + "\n"
    return table_str

def extract_answer(text):
    """Handles both numeric answers and categorical 'yes/no' responses."""
    text_lower = text.lower().strip()
    if 'yes' in text_lower:
        return 1.0
    if 'no' in text_lower:
        return 0.0
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text.replace(',', ''))
    return float(numbers[-1]) if numbers else None

def get_canonical_value(val):
    """Converts gold answers to a float for comparison."""
    if str(val).lower().strip() == 'yes':
        return 1.0
    if str(val).lower().strip() == 'no':
        return 0.0
    try:
        return float(str(val).replace(',', '').replace('%', ''))
    except ValueError:
        return None

def build_prompt(entry):
    table_str = format_table(entry['table'])
    context = " ".join(entry['pre_text'][-3:])
    question = entry['qa']['question']
    return f"""You are a financial analyst. Given the table and context below, answer the question by writing a step-by-step reasoning program.

Table:
{table_str}

Context:
{context}

Question:
{question}

First write your reasoning steps, then give the final numerical answer."""

def run_evaluation(sample_data, num_examples=100):
    correct = 0
    results = []
    for i in tqdm(range(len(sample_data))):
        entry = sample_data[i]
        prompt = build_prompt(entry)

        # Inference
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(**model_inputs, max_new_tokens=256, do_sample=False)
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        assistant_output = response.split("assistant\n")[-1]
        predicted_ans = extract_answer(assistant_output)
        gold_ans = get_canonical_value(entry['qa']['exe_ans'])

        if predicted_ans is not None and gold_ans is not None:
            if abs(predicted_ans - gold_ans) < 0.1:
                correct += 1

    return correct / len(sample_data)

all_accuracies = []

for i in range(1, 6):
    print(f"\n--- Running Validation Set {i} ---")
    random_sample = random.sample(data, 100)

    accuracy = run_evaluation(random_sample, num_examples=100)
    all_accuracies.append(accuracy)
    print(f"Set {i} Execution Accuracy: {accuracy * 100:.2f}%")

print("\n" + "="*30)
print("FINAL VARIANCE RESULTS")
print("="*30)
for idx, acc in enumerate(all_accuracies):
    print(f"Set {idx+1}: {acc*100:.2f}%")
print(f"Average Accuracy: {(sum(all_accuracies)/5)*100:.2f}%")


--- Running Validation Set 1 ---


100%|██████████| 100/100 [15:16<00:00,  9.17s/it]


Set 1 Execution Accuracy: 7.00%

--- Running Validation Set 2 ---


100%|██████████| 100/100 [14:33<00:00,  8.74s/it]


Set 2 Execution Accuracy: 9.00%

--- Running Validation Set 3 ---


...

Set 3 Execution Accuracy: 8.00%

--- Running Validation Set 4 ---


100%|██████████| 100/100 [14:23<00:00,  8.63s/it]


Set 4 Execution Accuracy: 4.00%

--- Running Validation Set 5 ---


100%|██████████| 100/100 [14:20<00:00,  8.60s/it]

Set 5 Execution Accuracy: 5.00%

FINAL VARIANCE RESULTS
Set 1: 7.00%
Set 2: 9.00%
Set 3: 8.00%
Set 4: 4.00%
Set 5: 5.00%
Average Accuracy: 6.60%


# Our Optimization

In [ ]:
def format_table(table):
    """Linearizes the table rows into a string format."""
    table_str = ""
    for row in table:
        table_str += " | ".join([str(cell) for cell in row]) + "\n"
    return table_str

def extract_answer(text):
    """Handles both numeric answers and categorical 'yes/no' responses."""
    text_lower = text.lower().strip()
    if 'yes' in text_lower:
        return 1.0
    if 'no' in text_lower:
        return 0.0
    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", text.replace(',', ''))
    return float(numbers[-1]) if numbers else None

def get_canonical_value(val):
    """Converts gold answers to a float for comparison."""
    if str(val).lower().strip() == 'yes':
        return 1.0
    if str(val).lower().strip() == 'no':
        return 0.0
    try:
        return float(str(val).replace(',', '').replace('%', ''))
    except ValueError:
        return None

In [ ]:
import re
import pandas as pd
from tqdm import tqdm

def build_layer3_final_prompt(entry):
    # Layer 3.2 Innovation: Inject 'Row:' tags to prevent spatial drift
    table = entry['table']
    headers = table[0]
    injected_rows = [headers]
    for row in table[1:]:
        row_label = row[0]
        # Reinforce the metric name at the start of every row
        injected_rows.append([f"Metric: {row_label}"] + row[1:])

    table_str = format_table(injected_rows)
    text_context = " ".join(entry['pre_text'] + entry['post_text'])
    question = entry['qa']['question']

    return f"""You are a financial calculator.
1. Use the Metric labels in the table to find the right numbers.
2. Perform the calculation.
3. End your response with 'Final Answer: ' followed by the decimal value.

CONTEXT:
{table_str}
{text_context}

QUESTION:
{question}

Analysis:"""

def extract_strict_answer(text):
    # Only look for the number immediately following the "Final Answer:" tag
    match = re.search(r"Final Answer:\s*([-+]?\d*\.\d+|\d+)", text)
    if match:
        return float(match.group(1).replace(',', ''))
    # Fallback to the old method if the tag is missing
    return extract_answer(text)

def run_layer3_final(data, num_examples=100):
    results = []
    correct_count = 0

    for i in tqdm(range(num_examples)):
        entry = data[i]
        prompt = build_layer3_final_prompt(entry)

        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=250, do_sample=False)
        response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

        logic = response.split("assistant\n")[-1]
        # NEW: Use strict extraction
        pred_ans = extract_strict_answer(logic)
        gold_ans = get_canonical_value(entry['qa']['exe_ans'])

        is_correct = False
        if pred_ans is not None and gold_ans is not None:
            # We also check if they gave a percentage (e.g. 9.6) instead of 0.096
            # and auto-convert to decimal for the evaluation check
            if pred_ans > 1.0 and gold_ans < 1.0:
                pred_ans = pred_ans / 100.0

            if abs(pred_ans - gold_ans) < 0.1:
                correct_count += 1
                is_correct = True

        results.append({
            "id": i,
            "is_correct": is_correct,
            "predicted_logic": logic,
            "gold_answer": gold_ans,
            "predicted_answer": pred_ans
        })

    return (correct_count / num_examples) * 100, results

# Execute the Final Layer 3 Test
final_accuracy, final_details = run_layer3_final(data, num_examples=100)
print(f"\nFinal Layer 3.2 Accuracy: {final_accuracy:.2f}%")

100%|██████████| 100/100 [13:10<00:00,  7.90s/it]


Final Layer 3.2 Accuracy: 20.00%


# Fine Tuning with Q-Lora

In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes

import pickle
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Environment ready and data reloaded!")

Environment ready and data reloaded!


In [ ]:
import json
import requests

url = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json"

response = requests.get(url)
data = json.loads(response.text)

entry = data[0]

print(f"--- Data Exploration (Entry 0) ---")
print(f"Question: {entry['qa']['question']}")
print(f"Program: {entry['qa']['program']}")
print(f"Execution Answer: {entry['qa']['exe_ans']}")
print(f"Gold Evidence (Indices): {entry['qa']['gold_inds']}")
print(f"Table Row 0: {entry['table'][0]}")
print(f"Pre-text sample: {entry['pre_text'][:2]}")
print()

In [17]:
import torch
import json
import re
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)

def get_canonical_value(val):
    try:
        return float(str(val).replace(',', '').replace('$', '').replace('%', '').strip())
    except: return None

def build_dsl_prompt(entry):
    # Formats linearized table for the 0.5B model
    table_str = "\n".join([" | ".join([str(cell) for cell in row]) for row in entry['table']])
    text_context = " ".join(entry['pre_text'] + entry['post_text'])
    return f"CONTEXT:\n{table_str}\n{text_context}\nQuestion: {entry['qa']['question']}\nProgram: "

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

train_subset = data[100:600]
training_list = []
for entry in train_subset:
    prompt = build_dsl_prompt(entry)
    # Target: Program | Answer
    target = f"{entry['qa']['program']} | Answer: {entry['qa']['exe_ans']}"
    training_list.append(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{target}<|im_end|>")

train_encodings = tokenizer(training_list, truncation=True, padding=True, max_length=1024, return_tensors="pt")

class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, encodings): self.encodings = encodings
    def __len__(self): return len(self.encodings.input_ids)
    def __getitem__(self, idx): return {k: v[idx] for k, v in self.encodings.items()}

train_loader = DataLoader(ManualDataset(train_encodings), batch_size=2, shuffle=True)

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
scaler = GradScaler()
num_steps = 300
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=40, num_training_steps=num_steps)

model.train()
pbar = tqdm(total=num_steps, desc="Training (500 Samples)")
step = 0

while step < num_steps:
    for batch in train_loader:
        if step >= num_steps: break
        batch = {k: v.to(model.device) for k, v in batch.items()}

        optimizer.zero_grad()
        with autocast():
            loss = model(**batch, labels=batch["input_ids"]).loss

        if not torch.isnan(loss):
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            step += 1
            pbar.update(1)
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

model.eval()
results = []
correct = 0

def robust_ans_extract(text):
    """Targets the value after the LAST 'Answer:' or the last number in the string."""
    if "Answer:" in text:
        parts = text.rsplit("Answer:", 1)
        target = parts[1]
    else:
        target = text

    nums = re.findall(r"[-+]?\d*\.\d+|\d+", target.replace(',', ''))
    return float(nums[-1]) if nums else None

print("\n--- Testing on Indices 0 to 100 ---")
for i in tqdm(range(0, 100)):
    entry = data[i]
    prompt = build_dsl_prompt(entry)
    inputs = tokenizer(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            repetition_penalty=1.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    p_prog = resp.split("|")[0].strip() if "|" in resp else resp
    p_ans = robust_ans_extract(resp)
    g_ans = get_canonical_value(entry['qa']['exe_ans'])

    is_correct = False
    if p_ans is not None and g_ans is not None:
        if p_ans > 1.0 and g_ans < 1.0: p_ans /= 100.0
        if abs(p_ans - g_ans) < 0.01:
            correct += 1
            is_correct = True

    results.append({
        "id": i,
        "is_correct": is_correct,
        "gold_p": entry['qa']['program'],
        "pred_p": p_prog,
        "gold_a": g_ans,
        "pred_a": p_ans,
        "raw": resp
    })

df_final = pd.DataFrame(results)
print(f"\nFinal Stable Accuracy: {(correct/100)*100:.2f}%")
df_final.to_csv('finqa_final_stable_fixed.csv', index=False)



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/tmp/ipykernel_13167/2839503301.py:80: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Training (500 Samples): 100%|██████████| 300/300 [19:37<00:00,  3.92s/it, loss=1.3474]
/tmp/ipykernel_13167/2839503301.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Training (500 Samples): 100%|██████████| 300/300 [0


--- Testing on Indices 0 to 100 ---



100%|██████████| 100/100 [06:24<00:00,  3.84s/it]


Final Stable Accuracy: 3.00%


In [18]:
# Corrected display logic
display(df_final.head(25).reset_index(drop=True))

,id,is_correct,gold_p,pred_p,gold_a,pred_a,raw
0,0,False,"subtract(5829, 5735)","subtract(187, 95) + subtract(32, 95) + subtrac...",94.00000,184.0000,"subtract(187, 95) + subtract(32, 95) + subtrac..."
1,1,False,"divide(8.1, 56.0)","divide(#1*100, #2)",0.14464,1.0000,"divide(#1*100, #2) | Answer: 1.0%"
2,2,False,"subtract(153.7, 139.9), divide(#0, 139.9)","subtract(139.9, 153.7), divide(#0, 139.9)",0.09864,-0.8800,"subtract(139.9, 153.7), divide(#0, 139.9) | An..."
3,3,False,"divide(121.4, 4187.8)","divide(total_purchase_price, 121.4)",0.02899,0.8500,"divide(total_purchase_price, 121.4) | Answer: ..."
4,4,False,"subtract(318.46, const_100), divide(#0, const_...","subtract(138.48, 132.04), divide(#1,#2)",1.11970,-0.4400,"subtract(138.48, 132.04), divide(#1,#2) | Answ..."
5,5,False,"subtract(92710000, 86842000), divide(#0, 86842...","subtract(100690000, 92710000), divide(#0, #0)",0.06757,-1.0880,"subtract(100690000, 92710000), divide(#0, #0) ..."
6,6,False,"divide(463, 4612)","divide(total_recourse_debt, total_recourse_debt)",0.10039,0.1500,"divide(total_recourse_debt, total_recourse_deb..."
7,7,False,"divide(301, 2575)","divide(249, 309)",0.11689,0.8370,"divide(249, 309) | Answer: 0.837"
8,8,False,"greater(286.61, 198.09)","took a look at it today, i think we were prett...",NaN,NaN,"took a look at it today, i think we were prett..."
9,9,False,"subtract(1136, 1171)","subtract(49, 44)",-35.00000,0.0500,"subtract(49, 44) | Answer: 5\n\nExplanation of..."
